# Assignment 04 — RAG Basics
## Agriculture Knowledge Assistant

### Objective
Build a Retrieval-Augmented Generation (RAG) pipeline over agriculture-related PDF documents. The system extracts document text, creates overlapping chunks, generates local embeddings, stores them in ChromaDB, retrieves relevant passages, and uses Gemini to answer questions grounded in the retrieved context.

### Workflow
1. Collect and inspect agriculture PDF documents.
2. Extract text, using OCR for scanned PDFs when necessary.
3. Split text into 500-character chunks with 50-character overlap.
4. Generate 384-dimensional embeddings with `all-MiniLM-L6-v2`.
5. Store chunks, embeddings, and source metadata in ChromaDB.
6. Retrieve the top 5 relevant chunks for a query.
7. Generate a grounded answer with source references.
8. Test refusal for questions not answered by the documents.

**Domain:** Agriculture  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Vector database:** ChromaDB  
**Answer-generation model:** Gemini

## 1. Install and import libraries

In [2]:
!pip install -q google-genai chromadb pypdf requests beautifulsoup4 sentence-transformers pymupdf pytesseract
!apt-get install -y tesseract-ocr

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [3]:
!pip install -q sentence-transformers

In [4]:
import os
import io
import numpy as np
import chromadb
import pypdf
import pymupdf
import pytesseract
from PIL import Image
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai

print("All required libraries imported successfully!")

All required libraries imported successfully!


## 2. Upload agriculture PDF documents

In [5]:
from google.colab import files

uploaded = files.upload()

print(f"\nTotal files uploaded: {len(uploaded)}")
for filename in uploaded:
    print(filename)

Saving BasmatiRiceReviseResige_.pdf to BasmatiRiceReviseResige_.pdf
Saving BulletinNCIPM2010_.pdf to BulletinNCIPM2010_.pdf
Saving cabbageegnlish_.pdf to cabbageegnlish_.pdf
Saving fruits_.pdf to fruits_.pdf
Saving Insect-pests-of-maize-and-their-management.pdf to Insect-pests-of-maize-and-their-management.pdf
Saving maize-diseases-and-their-management.pdf to maize-diseases-and-their-management.pdf
Saving PestofPigeonpea_.pdf to PestofPigeonpea_.pdf
Saving Selected-Agricultural-Technologies-A Compendium.pdf to Selected-Agricultural-Technologies-A Compendium.pdf
Saving Swarming_caterpillar_.pdf to Swarming_caterpillar_.pdf
Saving VegetableCropsenglish_.pdf to VegetableCropsenglish_.pdf

Total files uploaded: 10
BasmatiRiceReviseResige_.pdf
BulletinNCIPM2010_.pdf
cabbageegnlish_.pdf
fruits_.pdf
Insect-pests-of-maize-and-their-management.pdf
maize-diseases-and-their-management.pdf
PestofPigeonpea_.pdf
Selected-Agricultural-Technologies-A Compendium.pdf
Swarming_caterpillar_.pdf
VegetableC

## 3. Extract text from PDFs

In [6]:
import os
from pypdf import PdfReader

documents = []

for filename in uploaded:
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)

        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

        documents.append({
            "filename": filename,
            "text": text,
            "pages": len(reader.pages)
        })

print("PDF extraction completed!\n")

for doc in documents:
    print(f"File: {doc['filename']}")
    print(f"Pages: {doc['pages']}")
    print(f"Extracted characters: {len(doc['text'])}")
    print("-" * 50)

PDF extraction completed!

File: BasmatiRiceReviseResige_.pdf
Pages: 2
Extracted characters: 7388
--------------------------------------------------
File: BulletinNCIPM2010_.pdf
Pages: 18
Extracted characters: 24273
--------------------------------------------------
File: cabbageegnlish_.pdf
Pages: 2
Extracted characters: 10405
--------------------------------------------------
File: fruits_.pdf
Pages: 78
Extracted characters: 126517
--------------------------------------------------
File: Insect-pests-of-maize-and-their-management.pdf
Pages: 2
Extracted characters: 0
--------------------------------------------------
File: maize-diseases-and-their-management.pdf
Pages: 2
Extracted characters: 0
--------------------------------------------------
File: PestofPigeonpea_.pdf
Pages: 114
Extracted characters: 143253
--------------------------------------------------
File: Selected-Agricultural-Technologies-A Compendium.pdf
Pages: 131
Extracted characters: 225532
----------------------------

### Inspect extracted text quality

In [7]:
for doc in documents:
    print("\n" + "=" * 70)
    print("FILE:", doc["filename"])
    print("=" * 70)
    print(doc["text"][:500])


FILE: BasmatiRiceReviseResige_.pdf
Developed by
ICAR-National Research Centre for 
Integrated Pest Management
New Delhi – 110012
INTEGRATED 
PEST MANAGEMENT IN
BASMATI RICE
IPM Strategy with 
Safe Pesticide 
Residue Rice Production 
ETL: 3-5 lesions/leaf early to late tillering or 2-5 neck 2
infected plants/m  at panicle initiation to booting.
Spray tebuconazole 25% WG @750 ml/ha or 
azoxystrobin 18.2% w/w + difenoconazole 11.4% w/w 
SC  @ 0.1%  or 1 ml / litre water or tebuconazole 50% + 
trifloxystrobin 25% WG@200 g/ha: Tricycl

FILE: BulletinNCIPM2010_.pdf
Good Agricultural Practices (GAP) for IPM in Protected Cultivation
 15
Technical Bulletin No. 23
Good Agricultural Practices (GAP)
 for IPM in Protected Cultivation
Naved Sabir, Balraj Singh*, M. Hasan*, R. Sumitha,
Sikha Deka, R.K. Tanwar, D.B. Ahuja , B.S. Tomar**,
O.M. Bambawale & E.M. Khah†
National Centre for Integrated Pest Management (ICAR),
LBS Building, Pusa Campus, New Delhi-110012
*Centre for Protected Cultivation Tech

This prints the first 500 characters from each document.

Why? Character counts alone don't guarantee good extraction. We need to check whether the extracted text is readable and meaningful, especially tables and formatting.

### Inspect a document with potentially noisy extraction

Inspect the questionable document

In [8]:
doc = next(
    d for d in documents
    if d["filename"].startswith("Selected-Agricultural")
)

print("Document:", doc["filename"])
print("Total characters:", len(doc["text"]))

for position in [0, 5000, 20000, 50000, 100000]:
    print(f"\n--- Position {position} ---")
    print(doc["text"][position:position+500])

Document: Selected-Agricultural-Technologies-A Compendium.pdf
Total characters: 225532

--- Position 0 ---
5e le rted
Ag ricu lru rs I T$ch lr$log ies
, ,, Afump*ndilrs,
lndion Council of Agriculturol ReseorchSelec*sd
&gnfi*w**axrm$ Yex$Nmm**6**s
* * *,&ffompe*xd$*pga?$eilec*e*
&gr$cwilturffi [ Yechmotosfies
**.ACom*pes?d$wrxr
ICAR
INDIAN COUNCIL OF AGRICUTTURAL RESEARCH
NEW DETHICompilolion ond Editing
Director (DIPA)
Chief Production Officer
Technicol Officer
MAY 2OO4
Dn. R. Pmsnm
ADG (Asril. Extn.)
A Cnnrnnvnnrr
V K Bnmrr
Asnor Srnsrnr
Published by A Chokrovorty, Director, Directoroie of lnformoii

--- Position 5000 ---
D RICE
Crop % Yield States for which recommended for cultivatlon
duration advantage over
(Days) local check
APHR_I
DRRH_1
APHR_2
MGR-I
CORH_z
ANTRH.I
KRH-I
KRH_2
cl{RH-3
Pant Sankar
Dhan-1
Sahyadri
Narendra
Sankar Dhan-Z
PHB_71
PA 5201 (2ooo)
HRI-120 (2001)
r30-135
125-130
120-125
1 10-1 15
120-125
115-120
120*125
130-135
125-130
115-120
125-r30
125-r30
130-135
125-1

## 4. OCR scanned documents

Some PDFs may contain scanned page images instead of selectable text. The following cells install and verify Tesseract OCR, then extract text from the two scanned maize PDFs.

In [9]:
import fitz
import pytesseract
from PIL import Image

print("PyMuPDF version:", fitz.__doc__.split()[1])
print("Tesseract version:", pytesseract.get_tesseract_version())
print("OCR libraries ready!")

PyMuPDF version: 2.2:


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Tesseract version: 5.3.4
OCR libraries ready!


OCR the two maize PDFs

In [10]:
import pymupdf
import pytesseract
from PIL import Image
import io

maize_files = [
    "Insect-pests-of-maize-and-their-management.pdf",
    "maize-diseases-and-their-management.pdf"
]

ocr_results = {}

for filename in maize_files:
    pdf = pymupdf.open(filename)
    extracted_text = []

    for page_num, page in enumerate(pdf):
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
        image = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(image, lang="eng")
        extracted_text.append(text)

    full_text = "\n".join(extracted_text)
    ocr_results[filename] = full_text

    print(f"\nFile: {filename}")
    print(f"Pages: {len(pdf)}")
    print(f"Extracted characters: {len(full_text)}")
    print("\nPreview:")
    print(full_text[:700])
    print("-" * 60)


File: Insect-pests-of-maize-and-their-management.pdf
Pages: 2
Extracted characters: 13545

Preview:
Integrated Pest Management Strategies

Sowing to six leaf stage

+ Timely and uniform sowing over larger area.

Ridge and furrow planting method to be followed.

Application of recommended dosage of NPK as basal dose.

Planting of 3-4 rows of napier grass/hybrid napier as trap crops.

Intercropping of maize with legumes, viz., pigeonpea, cowpea, black gram,

kidney bean etc. in2: 1 to 4: 1 ratio.

+ Erection of bird perches @ 10/acre to encourage natural FAW predation by birds.

+ Installation of pheromone traps @ 4/acre soon after sowing and monitor moth
catches.

+ Adoption of clean cultivation to eliminate possible alternate hosts.

* Crushing of egg masses and larvae.

¢ Application of s
------------------------------------------------------------

File: maize-diseases-and-their-management.pdf
Pages: 2
Extracted characters: 10882

Preview:
Ploughing the field deeply during April and

### Add OCR text to the document collection

In [11]:
for filename, text in ocr_results.items():
    for doc in documents:
        if doc["filename"].startswith(filename):
            doc["text"] = text
            doc["extraction_method"] = "OCR"
            break

print("Updated document collection:\n")

for doc in documents:
    method = doc.get("extraction_method", "Native PDF")
    print(
        f"{doc['filename']} | "
        f"{len(doc['text'])} characters | "
        f"{method}"
    )

Updated document collection:

BasmatiRiceReviseResige_.pdf | 7388 characters | Native PDF
BulletinNCIPM2010_.pdf | 24273 characters | Native PDF
cabbageegnlish_.pdf | 10405 characters | Native PDF
fruits_.pdf | 126517 characters | Native PDF
Insect-pests-of-maize-and-their-management.pdf | 13545 characters | OCR
maize-diseases-and-their-management.pdf | 10882 characters | OCR
PestofPigeonpea_.pdf | 143253 characters | Native PDF
Selected-Agricultural-Technologies-A Compendium.pdf | 225532 characters | Native PDF
Swarming_caterpillar_.pdf | 32765 characters | Native PDF
VegetableCropsenglish_.pdf | 67521 characters | Native PDF


## 5. Chunk the extracted documents

Each document is split into fixed-size chunks of approximately **500 characters**, with **50 characters of overlap**. The overlap helps preserve context across chunk boundaries.

In [12]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "filename": doc["filename"],
            "chunk": chunk_id,
            "text": chunk
        })

print("Chunking completed!")
print("Total documents:", len(documents))
print("Total chunks:", len(all_chunks))

print("\nSample chunk:")
print(all_chunks[0])

Chunking completed!
Total documents: 10
Total chunks: 1478

Sample chunk:
{'filename': 'BasmatiRiceReviseResige_.pdf', 'chunk': 0, 'text': 'Developed by\nICAR-National Research Centre for \nIntegrated Pest Management\nNew Delhi – 110012\nINTEGRATED \nPEST MANAGEMENT IN\nBASMATI RICE\nIPM Strategy with \nSafe Pesticide \nResidue Rice Production \nETL: 3-5 lesions/leaf early to late tillering or 2-5 neck 2\ninfected plants/m  at panicle initiation to booting.\nSpray tebuconazole 25% WG @750 ml/ha or \nazoxystrobin 18.2% w/w + difenoconazole 11.4% w/w \nSC  @ 0.1%  or 1 ml / litre water or tebuconazole 50% + \ntrifloxystrobin 25% WG@200 g/ha: Tricycl'}


**Chunking configuration**

| Parameter | Value |
|---|---:|
| Chunk size | 500 characters |
| Overlap | 50 characters |
| Step size | 450 characters |

chunking configuration is:

Parameter	| Value

Chunk size | 500 characters

Overlap | 50 characters

Step size | 450 characters

Total chunks | 1,478

## 6. Generate local embeddings

The Gemini embedding API free-tier daily quota was reached during development. To complete the assignment without API quota restrictions, this notebook uses the local Sentence Transformers model `all-MiniLM-L6-v2` for **all** chunks and queries. Using one model consistently ensures that document and query vectors share the same embedding space.

In [13]:
!pip install -q sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [15]:
import numpy as np

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Total embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Total embeddings: 1478
Embedding dimensions: (1478, 384)


### Save the embeddings

Save the embeddings

In [16]:
import numpy as np

np.save("agriculture_embeddings.npy", all_embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


## 7. Create the ChromaDB vector store

ChromaDB stores each chunk together with its embedding and metadata. The persistent client keeps the vector store on disk in the Colab working directory.

In [17]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Recreate the collection so this notebook can be rerun cleanly.
try:
    chroma_client.delete_collection("agriculture_knowledge")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="agriculture_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created!")

ChromaDB collection created!


### Insert chunks, embeddings, and source metadata

In [18]:
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=[chunk["text"] for chunk in all_chunks],
    embeddings=all_embeddings.tolist(),
    metadatas=[
        {
            "filename": chunk["filename"],
            "chunk_number": chunk["chunk"]
        }
        for chunk in all_chunks
    ]
)

print("Total records in ChromaDB:", collection.count())

Total records in ChromaDB: 1478


## 8. Retrieve the top 5 relevant chunks

In [19]:
query = "What are the common insect pests affecting maize crops?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

for i in range(5):
    print(f"\nResult {i+1}")
    print("Source:", results["metadatas"][0][i]["filename"])
    print("Chunk:", results["metadatas"][0][i]["chunk_number"])
    print("Distance:", round(results["distances"][0][i], 4))
    print("Text:", results["documents"][0][i][:300])


Result 1
Source: Insect-pests-of-maize-and-their-management.pdf
Chunk: 13
Distance: 0.3082
Text: the
productivity of maize (3.1 t/ha) is much lower than the world average (5.62 t/ha).
Insect pests are one of the major factors responsible for low productivity. In the
context of India the major insect pests of maize which cause economic damage are
spotted stem borer [Chilo partellus (Swinhoe)], p

Result 2
Source: Insect-pests-of-maize-and-their-management.pdf
Chunk: 12
Distance: 0.316
Text: mi Soujanya, S. B. Suby, Jawala Jindal,
J.C. Sekhar, Ph. Romen Sharma and H.S.Jat

——

gor

7s:

Insect pests of maize
and
their management

ICAR - Indian Institute of Maize Research
Ludhiana-141 004, Punjab

Maize (Zea mays L.) is the third most important cereal crop after rice and wheat being
grow

Result 3
Source: cabbageegnlish_.pdf
Chunk: 0
Distance: 0.3353
Text: C
ole crops are attacked by a number of important insect
pests and diseases like diamond back moth, Plutella
xylostella; tobacco cate

## 9. Inspect retrieval quality

The query returns the five closest chunks according to cosine distance. **Lower distance indicates greater similarity.** Inspect the returned text and source metadata to judge whether the results answer the query; semantic similarity does not guarantee that every retrieved chunk is relevant.

Rank | Source | Distance | Relevance

1	| Maize | insect pests | 0.3082 | Excellent

2 | Maize | insect pests | 0.3160 | Excellent

3 | Cabbage | pests | 0.3353 | Less relevant

4 | Maize | insect pests | 0.3839 | Relevant

5 | Vegetable | crops | 0.4082 | Partially relevant

## 10. Generate a grounded answer with Gemini

The local embedding model is used for retrieval, while Gemini generates the natural-language answer. Source references are printed directly from ChromaDB metadata rather than asking the language model to invent citation details.

In [21]:
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [22]:
query = "What are the major insect pests affecting maize in India?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

query_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

context = "\n\n".join(query_results["documents"][0])

prompt = f"""
You are an agriculture knowledge assistant.

Answer the question using ONLY the provided context.
If the answer is unavailable, reply exactly:
I don't know from these documents.

Do not generate source citations yourself.

Context:
{context}

Question:
{query}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

print("Answer:\n")
print(response.text)

print("\nSources:")
for metadata in query_results["metadatas"][0]:
    print(f"- {metadata['filename']} — Chunk {metadata['chunk_number']}")

Answer:

Based on the provided documents, the major insect pests of maize in India that cause economic damage are:

* Spotted stem borer [*Chilo partellus* (Swinhoe)]
* Pink stem borer [*Sesamia inferens* Walker]
* Shoot fly [*Atherigona* spp.]
* Fall armyworm [*Spodoptera frugiperda* (J. E. Smith)] (noted as a recent invasive pest)

Sources:
- Insect-pests-of-maize-and-their-management.pdf — Chunk 13
- Insect-pests-of-maize-and-their-management.pdf — Chunk 12
- cabbageegnlish_.pdf — Chunk 0
- fruits_.pdf — Chunk 281
- Insect-pests-of-maize-and-their-management.pdf — Chunk 19


## 11. Test refusal for unsupported questions

Ask a question that is outside the agriculture documents. The expected response is exactly:

`I don't know from these documents.`

In [23]:
unsupported_query = "Who invented the telephone?"

unsupported_embedding = embedding_model.encode(
    [unsupported_query],
    normalize_embeddings=True
)[0]

unsupported_results = collection.query(
    query_embeddings=[unsupported_embedding.tolist()],
    n_results=5
)

unsupported_context = "\n\n".join(
    unsupported_results["documents"][0]
)

refusal_prompt = f"""
You are an agriculture knowledge assistant.

Answer using ONLY the provided context.
If the answer is not available in the context, reply exactly:
I don't know from these documents.

Context:
{unsupported_context}

Question:
{unsupported_query}
"""

refusal_response = client.models.generate_content(
    model=MODEL,
    contents=refusal_prompt
)

print(refusal_response.text)

I don't know from these documents.


**Observation:** Retrieval returned highly relevant maize-pest chunks at the top, but some unrelated crop-pest chunks also appeared. This indicates that retrieval quality can be improved further through better chunking, filtering, or reranking.

13. Inspect your document collection

In [24]:
print("Total documents:", len(documents))
print("\nDocument names:")

for doc in documents:
    print("-", doc["filename"])

print("\nTotal chunks:", len(all_chunks))
print("\nSample chunks from each document:")

for filename in set(chunk["filename"] for chunk in all_chunks):
    print(f"\n{'=' * 70}")
    print("DOCUMENT:", filename)

    sample = next(
        chunk["text"]
        for chunk in all_chunks
        if chunk["filename"] == filename
    )

    print(sample[:500])

Total documents: 10

Document names:
- BasmatiRiceReviseResige_.pdf
- BulletinNCIPM2010_.pdf
- cabbageegnlish_.pdf
- fruits_.pdf
- Insect-pests-of-maize-and-their-management.pdf
- maize-diseases-and-their-management.pdf
- PestofPigeonpea_.pdf
- Selected-Agricultural-Technologies-A Compendium.pdf
- Swarming_caterpillar_.pdf
- VegetableCropsenglish_.pdf

Total chunks: 1478

Sample chunks from each document:

DOCUMENT: fruits_.pdf
PESTS OF FRUITS
(Banana, Mango and Pomegranate) :
‘E’ Pest Surveillance and Pest Management Advisory
Implementation : Commissionerate of Agriculture, Pune, Maharashtra
Project Coordination : National Centre for Integrated Pest Management,
New Delhi
‘e’ Pest Surveillance Coordination : Department of Horticulture, Government of
Maharashtra, Pune, MS
Technical Manual - 31NCIPM
ii
Contributors
D.B. Ahuja, Niranjan Singh, A.K. Kanojia : National Centre for Integrated Pest
R.V. Singh, R. Sumitha, Hera

DOCUMENT: Insect-pests-of-maize-and-their-management.pdf
Integrate

14. Prepare the 15 evaluation questions

In [25]:
evaluation_questions = [
    # 10 single-document questions
    {"id": 1, "type": "single", "question": "What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?"},
    {"id": 2, "type": "single", "question": "Which trap crops are recommended for managing fall armyworm in maize?"},
    {"id": 3, "type": "single", "question": "Which leguminous crops are suggested for intercropping with maize?"},
    {"id": 4, "type": "single", "question": "How many bird perches per acre are recommended in maize fields?"},
    {"id": 5, "type": "single", "question": "What practices are recommended for managing infected maize crop residues?"},
    {"id": 6, "type": "single", "question": "Which insect pests commonly attack cabbage and other cole crops?"},
    {"id": 7, "type": "single", "question": "Which fruit crops are covered in the fruit pest surveillance and management advisory?"},
    {"id": 8, "type": "single", "question": "What is the scientific name of the rice swarming caterpillar?"},
    {"id": 9, "type": "single", "question": "What is the main focus of the integrated pest management strategy for Basmati rice?"},
    {"id": 10, "type": "single", "question": "What is the purpose of the Good Agricultural Practices guidelines for IPM in protected cultivation?"},

    # 3 questions requiring two documents
    {"id": 11, "type": "multi", "question": "Compare the recommended pest management approaches for maize and Basmati rice using their respective guidelines."},
    {"id": 12, "type": "multi", "question": "What crop protection information is provided for maize diseases and rice swarming caterpillar management?"},
    {"id": 13, "type": "multi", "question": "Compare the pest management guidance for cabbage and pigeonpea using the respective documents."},

    # 2 unsupported questions
    {"id": 14, "type": "unsupported", "question": "Who invented the telephone?"},
    {"id": 15, "type": "unsupported", "question": "What is the current market price of gold in Chennai?"}
]

print("Total questions:", len(evaluation_questions))

for item in evaluation_questions:
    query_embedding = embedding_model.encode(
        [item["question"]],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=2
    )

    print("\n" + "=" * 80)
    print(f"Q{item['id']} | {item['type'].upper()}")
    print("Question:", item["question"])

    for i, text in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        print(f"\nResult {i+1}: {metadata['filename']}")
        print(text[:700])

Total questions: 15

Q1 | SINGLE
Question: What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?

Result 1: Insect-pests-of-maize-and-their-management.pdf
Integrated Pest Management Strategies

Sowing to six leaf stage

+ Timely and uniform sowing over larger area.

Ridge and furrow planting method to be followed.

Application of recommended dosage of NPK as basal dose.

Planting of 3-4 rows of napier grass/hybrid napier as trap crops.

Intercropping of maize with legumes, viz., pigeonpea, cowpea, black gram,

kidney bean etc. in2: 1 to 4: 1 ratio.

+ Erection of bird perches @ 10/acre to encourage natural FAW predation by birds.

+ Installation o

Result 2: Insect-pests-of-maize-and-their-management.pdf
mi Soujanya, S. B. Suby, Jawala Jindal,
J.C. Sekhar, Ph. Romen Sharma and H.S.Jat

——

gor

7s:

Insect pests of maize
and
their management

ICAR - Indian Institute of Maize Research
Ludhiana-141 004, Punjab

Maize (Zea mays L.) is the t

Current RAG system retrieves some correct passages, but there are several misses:

Question | Observation

Q1 | Correct document and relevant passage

Q2 | Retrieved the maize document, but not the passage identifying trap crops

Q3 | Retrieved a related maize passage, but not the expected list of legumes

Q4 | Correct information appears in Result 2, but Result 1 is irrelevant

Q5–Q8 | Mostly correct retrieval

Q9–Q10 | Retrieved the right document, but not enough information to answer confidently

Q11–Q13 | Multi-document retrieval is failing to retrieve both required sources

Q14–Q15 | Irrelevant results, as expected for unsupported questions

Important: We shouldn't count a question as successfully retrieved just because the correct PDF appears. The retrieved passage must actually contain evidence that answers the question.

15. Test Top-5 Retrievals of questions before change

In [28]:
baseline_results = []

for item in evaluation_questions:

    query_embedding = embedding_model.encode(
        [item["question"]],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    retrieved_docs = results["metadatas"][0]
    retrieved_texts = results["documents"][0]

    baseline_results.append({
        "id": item["id"],
        "type": item["type"],
        "question": item["question"],
        "documents": [
            metadata["filename"] for metadata in retrieved_docs
        ],
        "passages": retrieved_texts
    })

    print("\n" + "=" * 80)
    print(f"Q{item['id']} | {item['type'].upper()}")
    print(item["question"])

    for rank, (metadata, passage) in enumerate(
        zip(retrieved_docs, retrieved_texts), start=1
    ):
        print(f"\nRank {rank}: {metadata['filename']}")
        print(passage[:350].replace("\n", " "))


Q1 | SINGLE
What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?

Rank 1: Insect-pests-of-maize-and-their-management.pdf
Integrated Pest Management Strategies  Sowing to six leaf stage  + Timely and uniform sowing over larger area.  Ridge and furrow planting method to be followed.  Application of recommended dosage of NPK as basal dose.  Planting of 3-4 rows of napier grass/hybrid napier as trap crops.  Intercropping of maize with legumes, viz., pigeonpea, cowpea, bl

Rank 2: Insect-pests-of-maize-and-their-management.pdf
mi Soujanya, S. B. Suby, Jawala Jindal, J.C. Sekhar, Ph. Romen Sharma and H.S.Jat  ——  gor  7s:  Insect pests of maize and their management  ICAR - Indian Institute of Maize Research Ludhiana-141 004, Punjab  Maize (Zea mays L.) is the third most important cereal crop after rice and wheat being grown in tropical, sub tropical and temperate regions 

Rank 3: VegetableCropsenglish_.pdf
Validated Integrated Pest Manageme

16. Calculating Retrieval Performance

In [29]:
gold_sources = {
    1: ["Insect-pests-of-maize-and-their-management.pdf"],
    2: ["Insect-pests-of-maize-and-their-management.pdf"],
    3: ["Insect-pests-of-maize-and-their-management.pdf"],
    4: ["Insect-pests-of-maize-and-their-management.pdf"],
    5: ["maize-diseases-and-their-management.pdf"],
    6: ["cabbageegnlish_.pdf"],
    7: ["fruits_.pdf"],
    8: ["Swarming_caterpillar_.pdf"],
    9: ["BasmatiRiceReviseResige_.pdf"],
    10: ["BulletinNCIPM2010_.pdf"],
    11: [
        "Insect-pests-of-maize-and-their-management.pdf",
        "BasmatiRiceReviseResige_.pdf"
    ],
    12: [
        "maize-diseases-and-their-management.pdf",
        "Swarming_caterpillar_.pdf"
    ],
    13: [
        "cabbageegnlish_.pdf",
        "PestofPigeonpea_.pdf"
    ],
    14: [],
    15: []
}

retrieval_scores = []

for result in baseline_results:
    qid = result["id"]
    expected = set(gold_sources[qid])
    retrieved = set(result["documents"])

    if not expected:
        # Unsupported questions: no expected source document
        score = 1 if len(retrieved) > 0 else 0
    else:
        score = len(expected & retrieved) / len(expected)

    retrieval_scores.append(score)

    print(
        f"Q{qid:02d} | "
        f"Expected: {len(expected)} | "
        f"Retrieved: {len(expected & retrieved)} | "
        f"Score: {score:.2f}"
    )

answerable_scores = retrieval_scores[:13]

print("\nAnswerable question source recall:")
print(f"{sum(answerable_scores) / len(answerable_scores):.2%}")

Q01 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q02 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q03 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q04 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q05 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q06 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q07 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q08 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q09 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q10 | Expected: 1 | Retrieved: 1 | Score: 1.00
Q11 | Expected: 2 | Retrieved: 1 | Score: 0.50
Q12 | Expected: 2 | Retrieved: 1 | Score: 0.50
Q13 | Expected: 2 | Retrieved: 1 | Score: 0.50
Q14 | Expected: 0 | Retrieved: 0 | Score: 1.00
Q15 | Expected: 0 | Retrieved: 0 | Score: 1.00

Answerable question source recall:
88.46%


top-5 source-level retrieval recall is 88.46% for the 13 answerable questions.

the results are:
- Q1–Q10: All 10 single-document questions retrieved the expected source.
- Q11–Q13: Each retrieved only 1 of the 2 expected documents. These multi-document questions are where retrieval needs improvement.
- Q14–Q15: Unsupported questions should be evaluated separately as refusal tests, not included in retrieval recall.

17. Defining expected answers for 15 questions

In [30]:
expected_answers = {
    1: "Recommended practices include timely and uniform sowing, ridge and furrow planting, basal fertilizer application, trap crops, intercropping, bird perches and pheromone traps.",
    2: "Napier grass or hybrid Napier can be planted as trap crops around maize fields.",
    3: "Pigeonpea, cowpea, black gram and kidney bean are suggested for intercropping with maize.",
    4: "10 bird perches per acre.",
    5: "Deep plough infected plant debris, remove and burn infected crop residues, and follow recommended crop sanitation practices.",
    6: "Diamondback moth, tobacco caterpillar, cabbage butterfly, head borer, painted bug and cabbage aphid.",
    7: "Banana, mango and pomegranate.",
    8: "Spodoptera mauritia.",
    9: "Integrated pest management for safe Basmati rice production with low pesticide residues.",
    10: "To promote environmentally, economically and socially sustainable on-farm practices that produce safe and quality products.",
    11: "Maize guidance includes crop management, trap crops, intercropping and monitoring practices. Basmati rice guidance focuses on integrated pest management and safe pesticide-residue production.",
    12: "Maize disease management includes removing and destroying infected crop residues and other preventive practices. Rice swarming caterpillar management addresses the pest Spodoptera mauritia.",
    13: "Cabbage guidance covers pests such as diamondback moth and cabbage aphid. Pigeonpea guidance covers crop-specific pest management, field sanitation, summer ploughing and seed treatment.",
    14: "UNSUPPORTED — the agriculture knowledge base does not contain this information.",
    15: "UNSUPPORTED — the agriculture knowledge base does not contain current gold prices."
}

print("Reference answers prepared:", len(expected_answers))

Reference answers prepared: 15


18. Checking the answer-generation setup

In [31]:
print("Gemini client:", type(client))
print("Baseline result structure:", baseline_results[0].keys())
print("\nSample baseline result:")
print(baseline_results[0])

Gemini client: <class 'google.genai.client.Client'>
Baseline result structure: dict_keys(['id', 'type', 'question', 'documents', 'passages'])

Sample baseline result:
{'id': 1, 'type': 'single', 'question': 'What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?', 'documents': ['Insect-pests-of-maize-and-their-management.pdf', 'Insect-pests-of-maize-and-their-management.pdf', 'VegetableCropsenglish_.pdf', 'maize-diseases-and-their-management.pdf', 'Insect-pests-of-maize-and-their-management.pdf'], 'passages': ['Integrated Pest Management Strategies\n\nSowing to six leaf stage\n\n+ Timely and uniform sowing over larger area.\n\nRidge and furrow planting method to be followed.\n\nApplication of recommended dosage of NPK as basal dose.\n\nPlanting of 3-4 rows of napier grass/hybrid napier as trap crops.\n\nIntercropping of maize with legumes, viz., pigeonpea, cowpea, black gram,\n\nkidney bean etc. in2: 1 to 4: 1 ratio.\n\n+ Erection of bird

Checking purpose(not assignment)

In [33]:
for name, value in globals().items():
    if "model" in name.lower() and isinstance(value, str):
        print(f"{name} = {value}")

MODEL = gemini-3.5-flash-lite


19. Generate answers for all 15 questions

In [34]:
generated_answers = []

for item in baseline_results:
    context = "\n\n".join(item["passages"])

    prompt = f"""
You are an agricultural knowledge assistant.
Answer the question using ONLY the supplied context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, respond:
  "I don't have enough information in the provided documents."
- Keep the answer concise.

Context:
{context}

Question:
{item["question"]}

Answer:
"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        answer = response.text.strip()
    except Exception as e:
        answer = f"GENERATION_ERROR: {str(e)}"

    generated_answers.append({
        "id": item["id"],
        "question": item["question"],
        "answer": answer,
        "expected": expected_answers[item["id"]]
    })

    print(f"\nQ{item['id']:02d}: {item['question']}")
    print("Generated:", answer)
    print("Expected:", expected_answers[item["id"]])
    print("-" * 80)


Q01: What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?
Generated: Based on the provided context, the integrated pest management practices recommended for maize from sowing to the six-leaf stage are:

* Timely and uniform sowing over a larger area.
* Following the ridge and furrow planting method.
* Application of the recommended dosage of NPK as a basal dose.
* Planting 3-4 rows of napier grass/hybrid napier as trap crops.
* Intercropping maize with legumes (such as pigeonpea, cowpea, black gram, kidney bean, etc.) in a 2:1 to 4:1 ratio.
* Erection of bird perches at 10/acre to encourage natural FAW predation by birds.
Expected: Recommended practices include timely and uniform sowing, ridge and furrow planting, basal fertilizer application, trap crops, intercropping, bird perches and pheromone traps.
--------------------------------------------------------------------------------

Q02: Which trap crops are recommended for managing f

## 20. Baseline Evaluation Results

- Test questions: 15
- Answerable questions: 13
- Unsupported questions: 2
- Top-5 source-level retrieval recall: 88.46%
- Manual answer-quality score: 63.33%

### Key Findings
- Single-document retrieval performed well.
- Multi-document retrieval struggled to retrieve all required sources.
- Some answerable questions resulted in unnecessary refusals.
- Unsupported questions were correctly refused.

### Limitations
The evaluation uses a small agriculture-focused document collection and
manual answer scoring. The results are an initial baseline, not a
comprehensive measure of RAG performance.

21. Chunk-size comparison (200, 500 and 1000-character chunks)

In [35]:
print("Document keys:", documents[0].keys())
print("Chunk keys:", all_chunks[0].keys())

Document keys: dict_keys(['filename', 'text', 'pages'])
Chunk keys: dict_keys(['filename', 'chunk', 'text'])


In [36]:
import numpy as np

# Chunking function
def create_chunks(text, chunk_size, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunk_sizes = [200, 500, 1000]
chunk_comparison = {}

for size in chunk_sizes:
    print(f"\nEvaluating chunk size: {size}")

    # Create chunks from all source documents
    test_chunks = []

    for doc in documents:
        pieces = create_chunks(doc["text"], size, overlap=50)

        for piece in pieces:
            test_chunks.append({
                "filename": doc["filename"],
                "text": piece
            })

    # Generate embeddings
    texts = [item["text"] for item in test_chunks]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    # Evaluate retrieval for all 15 questions
    results = []

    for q in evaluation_questions:
        query_embedding = embedding_model.encode(
            [q["question"]],
            normalize_embeddings=True
        )[0]

        scores = np.dot(embeddings, query_embedding)
        top_indices = np.argsort(scores)[::-1][:5]

        retrieved_sources = {
            test_chunks[i]["filename"] for i in top_indices
        }

        expected_sources = set(gold_sources[q["id"]])

        if expected_sources:
            score = (
                len(expected_sources & retrieved_sources)
                / len(expected_sources)
            )
        else:
            score = None

        results.append({
            "id": q["id"],
            "retrieved_sources": list(retrieved_sources),
            "score": score
        })

    answerable_scores = [
        r["score"] for r in results if r["score"] is not None
    ]

    recall = np.mean(answerable_scores)

    chunk_comparison[size] = {
        "total_chunks": len(test_chunks),
        "recall": recall,
        "results": results
    }

    print("Total chunks:", len(test_chunks))
    print(f"Source-level recall: {recall:.2%}")

print("\nCHUNK SIZE COMPARISON")
for size, result in chunk_comparison.items():
    print(
        f"{size} characters | "
        f"{result['total_chunks']} chunks | "
        f"Recall: {result['recall']:.2%}"
    )


Evaluating chunk size: 200


Batches:   0%|          | 0/139 [00:00<?, ?it/s]

Total chunks: 4420
Source-level recall: 88.46%

Evaluating chunk size: 500


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Total chunks: 1478
Source-level recall: 88.46%

Evaluating chunk size: 1000


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Total chunks: 702
Source-level recall: 88.46%

CHUNK SIZE COMPARISON
200 characters | 4420 chunks | Recall: 88.46%
500 characters | 1478 chunks | Recall: 88.46%
1000 characters | 702 chunks | Recall: 88.46%


## 22. Chunk Size Comparison

Tested chunk sizes: 200, 500 and 1000 characters.
Overlap: 50 characters.
Retrieval method: Dense vector search.
Top-K: 5.

| Chunk Size | Total Chunks | Source Recall |
|---|---:|---:|
| 200 | 4420 | 88.46% |
| 500 | 1478 | 88.46% |
| 1000 | 702 | 88.46% |

### Conclusion
All three configurations achieved the same source-level recall.
The 500-character chunk size was selected as a balance between
context size and the number of chunks to process.

This experiment measures source-level recall, not answer quality
or retrieval latency.

23. Hybrid search (BM25 + vector)

In [37]:
!pip -q install rank-bm25

In [38]:
from rank_bm25 import BM25Okapi
import numpy as np

# Use the existing 500-character chunks
hybrid_chunks = all_chunks

# Prepare documents for BM25
tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in hybrid_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

# Generate embeddings for vector retrieval
hybrid_texts = [chunk["text"] for chunk in hybrid_chunks]

hybrid_embeddings = embedding_model.encode(
    hybrid_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Hybrid search setup completed!")
print("BM25 documents:", len(tokenized_corpus))
print("Vector embeddings:", hybrid_embeddings.shape)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Hybrid search setup completed!
BM25 documents: 1478
Vector embeddings: (1478, 384)


24. Implement Reciprocal Rank Fusion (RRF)

In [39]:
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}

    for ranking in rankings:
        for rank, doc_index in enumerate(ranking, start=1):
            scores[doc_index] = scores.get(doc_index, 0) + 1 / (k + rank)

    return sorted(scores, key=scores.get, reverse=True)


def hybrid_search(question, top_k=5):
    # 1. BM25 retrieval
    query_tokens = question.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top20 = np.argsort(bm25_scores)[::-1][:20]

    # 2. Vector retrieval
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    vector_scores = np.dot(hybrid_embeddings, query_embedding)
    vector_top20 = np.argsort(vector_scores)[::-1][:20]

    # 3. Combine rankings using RRF
    fused_ranking = reciprocal_rank_fusion(
        [bm25_top20, vector_top20]
    )

    return [
        hybrid_chunks[i] for i in fused_ranking[:top_k]
    ]


# Test one question
test_question = evaluation_questions[0]["question"]
hybrid_results = hybrid_search(test_question)

print("Question:", test_question)
print("\nTop 5 Hybrid Results:")

for rank, result in enumerate(hybrid_results, start=1):
    print(f"\nRank {rank}: {result['filename']}")
    print(result["text"][:300])

Question: What integrated pest management practices are recommended for maize from sowing to the six-leaf stage?

Top 5 Hybrid Results:

Rank 1: Insect-pests-of-maize-and-their-management.pdf
Integrated Pest Management Strategies

Sowing to six leaf stage

+ Timely and uniform sowing over larger area.

Ridge and furrow planting method to be followed.

Application of recommended dosage of NPK as basal dose.

Planting of 3-4 rows of napier grass/hybrid napier as trap crops.

Intercropping 

Rank 2: Insect-pests-of-maize-and-their-management.pdf
mi Soujanya, S. B. Suby, Jawala Jindal,
J.C. Sekhar, Ph. Romen Sharma and H.S.Jat

——

gor

7s:

Insect pests of maize
and
their management

ICAR - Indian Institute of Maize Research
Ludhiana-141 004, Punjab

Maize (Zea mays L.) is the third most important cereal crop after rice and wheat being
grow

Rank 3: maize-diseases-and-their-management.pdf
Ploughing the field deeply during April and May, with 15-day intervals, helps
to bury the infected pl

25. Evaluate hybrid search on all 15 questions

In [40]:
import time
import numpy as np

hybrid_evaluation_results = []
hybrid_latencies = []

for q in evaluation_questions:
    start_time = time.perf_counter()

    results = hybrid_search(q["question"], top_k=5)

    latency = time.perf_counter() - start_time
    hybrid_latencies.append(latency)

    retrieved_sources = {
        item["filename"] for item in results
    }

    expected_sources = set(gold_sources[q["id"]])

    if expected_sources:
        score = (
            len(expected_sources & retrieved_sources)
            / len(expected_sources)
        )
    else:
        score = None

    hybrid_evaluation_results.append({
        "id": q["id"],
        "question": q["question"],
        "retrieved_sources": list(retrieved_sources),
        "score": score,
        "latency": latency
    })

    print(
        f"Q{q['id']:02d} | "
        f"Recall: {score if score is not None else 'N/A'} | "
        f"Latency: {latency:.3f}s"
    )

answerable_scores = [
    r["score"] for r in hybrid_evaluation_results
    if r["score"] is not None
]

hybrid_recall = np.mean(answerable_scores)
hybrid_avg_latency = np.mean(hybrid_latencies)

print("\nHYBRID SEARCH SUMMARY")
print(f"Source-level recall: {hybrid_recall:.2%}")
print(f"Average latency: {hybrid_avg_latency:.3f} seconds")
print(f"Questions evaluated: {len(evaluation_questions)}")

Q01 | Recall: 1.0 | Latency: 0.053s
Q02 | Recall: 1.0 | Latency: 0.045s
Q03 | Recall: 1.0 | Latency: 0.038s
Q04 | Recall: 1.0 | Latency: 0.067s
Q05 | Recall: 1.0 | Latency: 0.060s
Q06 | Recall: 1.0 | Latency: 0.045s
Q07 | Recall: 1.0 | Latency: 0.042s
Q08 | Recall: 1.0 | Latency: 0.040s
Q09 | Recall: 1.0 | Latency: 0.042s
Q10 | Recall: 1.0 | Latency: 0.053s
Q11 | Recall: 1.0 | Latency: 0.083s
Q12 | Recall: 0.5 | Latency: 0.099s
Q13 | Recall: 0.5 | Latency: 0.071s
Q14 | Recall: N/A | Latency: 0.040s
Q15 | Recall: N/A | Latency: 0.040s

HYBRID SEARCH SUMMARY
Source-level recall: 92.31%
Average latency: 0.055 seconds
Questions evaluated: 15


## 26. Hybrid Search Evaluation

### Configuration
- Keyword retrieval: BM25
- Semantic retrieval: SentenceTransformer embeddings
- Fusion: Reciprocal Rank Fusion (RRF)
- RRF constant: 60
- Candidate documents: Top 20 from each retriever
- Final results: Top 5

### Results

| Metric | Vector-only | Hybrid |
|---|---:|---:|
| Source recall | 88.46% | 92.31% |
| Average latency | Not recorded | 0.055 sec |

### Findings
Hybrid search improved source-level recall by 3.85 percentage
points. It successfully retrieved both expected documents for Q11.
However, Q12 and Q13 still retrieved only one of their two
expected sources.

### Conclusion
Combining keyword-based and semantic retrieval improved
the baseline retrieval performance on this test set.

27. Cross-encoder reranking

In [41]:
!pip -q install sentence-transformers

In [42]:
from sentence_transformers import CrossEncoder
import numpy as np
import time

# Load cross-encoder model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-encoder loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded successfully!


In [43]:
def reranked_search(question, candidate_k=20, top_k=5):
    # Retrieve top 20 candidates using vector search
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    scores = np.dot(hybrid_embeddings, query_embedding)
    candidate_indices = np.argsort(scores)[::-1][:candidate_k]

    candidates = [hybrid_chunks[i] for i in candidate_indices]

    # Cross-encoder scores query-document pairs jointly
    pairs = [
        [question, item["text"]]
        for item in candidates
    ]

    rerank_scores = reranker.predict(pairs)

    # Sort candidates by cross-encoder score
    ranked_indices = np.argsort(rerank_scores)[::-1][:top_k]

    return [candidates[i] for i in ranked_indices]


# Test with Q01
reranked_results = reranked_search(evaluation_questions[0]["question"])

for rank, item in enumerate(reranked_results, start=1):
    print(f"\nRank {rank}: {item['filename']}")
    print(item["text"][:300])


Rank 1: Insect-pests-of-maize-and-their-management.pdf
Integrated Pest Management Strategies

Sowing to six leaf stage

+ Timely and uniform sowing over larger area.

Ridge and furrow planting method to be followed.

Application of recommended dosage of NPK as basal dose.

Planting of 3-4 rows of napier grass/hybrid napier as trap crops.

Intercropping 

Rank 2: maize-diseases-and-their-management.pdf
Ploughing the field deeply during April and May, with 15-day intervals, helps
to bury the infected plant debris.

Remove and burn the infected crop residues from the field

Avoid water stress especially during the flowering and grain filling stages.
Grow maize hybrids that are recommended for their 

Rank 3: PestofPigeonpea_.pdf
of sustainable agriculture, includes following pest
management methods e.g., monitoring, cultural, physical or mechanical
threshold assessment, biological, and chemical control, which are compatible
with each other. Like others crops, an urgent need was felt so t

28. Evaluate cross-encoder reranking

In [44]:
reranker_results = []
reranker_latencies = []

for q in evaluation_questions:
    start_time = time.perf_counter()

    results = reranked_search(
        q["question"],
        candidate_k=20,
        top_k=5
    )

    latency = time.perf_counter() - start_time
    reranker_latencies.append(latency)

    retrieved_sources = {
        item["filename"] for item in results
    }

    expected_sources = set(gold_sources[q["id"]])

    if expected_sources:
        score = (
            len(expected_sources & retrieved_sources)
            / len(expected_sources)
        )
    else:
        score = None

    reranker_results.append({
        "id": q["id"],
        "retrieved_sources": list(retrieved_sources),
        "score": score,
        "latency": latency
    })

    print(
        f"Q{q['id']:02d} | "
        f"Recall: {score if score is not None else 'N/A'} | "
        f"Latency: {latency:.3f}s"
    )

answerable_scores = [
    r["score"] for r in reranker_results
    if r["score"] is not None
]

reranker_recall = np.mean(answerable_scores)
reranker_avg_latency = np.mean(reranker_latencies)

print("\nCROSS-ENCODER SUMMARY")
print(f"Source-level recall: {reranker_recall:.2%}")
print(f"Average latency: {reranker_avg_latency:.3f} seconds")
print("Candidate retrieval: Top 20")
print("Final reranked results: Top 5")

Q01 | Recall: 1.0 | Latency: 5.089s
Q02 | Recall: 1.0 | Latency: 2.344s
Q03 | Recall: 1.0 | Latency: 2.045s
Q04 | Recall: 1.0 | Latency: 2.587s
Q05 | Recall: 1.0 | Latency: 1.733s
Q06 | Recall: 1.0 | Latency: 2.443s
Q07 | Recall: 1.0 | Latency: 1.288s
Q08 | Recall: 1.0 | Latency: 1.482s
Q09 | Recall: 1.0 | Latency: 1.629s
Q10 | Recall: 1.0 | Latency: 1.587s
Q11 | Recall: 1.0 | Latency: 1.608s
Q12 | Recall: 0.5 | Latency: 1.522s
Q13 | Recall: 0.5 | Latency: 3.695s
Q14 | Recall: N/A | Latency: 3.119s
Q15 | Recall: N/A | Latency: 2.646s

CROSS-ENCODER SUMMARY
Source-level recall: 92.31%
Average latency: 2.321 seconds
Candidate retrieval: Top 20
Final reranked results: Top 5


## Final Retrieval Strategy Comparison

| Method | Top-K | Source Recall | Avg Latency |
|---|---:|---:|---:|
| Vector-only | 5 | 88.46% | Not recorded |
| Hybrid BM25 + Vector | 5 | 92.31% | 0.055 sec |
| Vector + Cross-encoder | 20 → 5 | 92.31% | 2.321 sec |

### Selected Configuration
Hybrid search using BM25 and vector retrieval with Reciprocal Rank Fusion.

### Justification
Hybrid search achieved the highest observed recall (tied with
cross-encoder reranking) while maintaining much lower latency.
The cross-encoder did not provide a measurable recall improvement
on this evaluation dataset.

### Remaining Weaknesses
Multi-document questions Q12 and Q13 achieved only 50% source
recall. Further improvements are needed for multi-document retrieval.

## Prompting vs RAG vs Fine-tuning

### Prompting
Useful for controlling answer format, tone, and instructions.
It does not independently provide access to private or external
knowledge.

### RAG
Best suited for this agriculture knowledge assistant because it
retrieves information from external agricultural documents and
provides context to the language model. Documents can be updated
without retraining the model.

### Fine-tuning
Useful for adapting model behaviour, domain-specific language,
or output patterns. It requires training data and computational
resources and is not the best way to keep changing agricultural
facts up to date.

### Recommendation
Use RAG with hybrid retrieval and a carefully designed prompt.
Fine-tuning can be considered later if domain-specific response
quality remains inadequate.

29. Test whether hybrid retrieval improves the answer quality, not just retrieval recall.

In [45]:
hybrid_generated_answers = []

for q in evaluation_questions:
    # Retrieve top 5 using hybrid search
    results = hybrid_search(q["question"], top_k=5)

    context = "\n\n".join(
        item["text"] for item in results
    )

    prompt = f"""
You are an agricultural knowledge assistant.
Answer using ONLY the supplied context.

Rules:
- Do not use outside knowledge.
- If the context is insufficient, say:
  "I don't have enough information in the provided documents."
- For questions requiring multiple sources, combine
  the available relevant information.
- Keep the answer concise.

Context:
{context}

Question:
{q["question"]}

Answer:
"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        answer = response.text.strip()
    except Exception as e:
        answer = f"GENERATION_ERROR: {str(e)}"

    hybrid_generated_answers.append({
        "id": q["id"],
        "question": q["question"],
        "answer": answer,
        "expected": expected_answers[q["id"]]
    })

    print(f"\nQ{q['id']:02d}")
    print("Generated:", answer)
    print("Expected:", expected_answers[q["id"]])
    print("-" * 80)


Q01
Generated: Based on the provided context, the integrated pest management practices recommended for maize from sowing to the six-leaf stage are:

* Timely and uniform sowing over a larger area.
* Following a ridge and furrow planting method.
* Applying the recommended dosage of NPK as a basal dose.
* Planting 3-4 rows of napier grass/hybrid napier as trap crops.
* Intercropping maize with legumes (such as pigeonpea, cowpea, black gram, kidney bean, etc.) in a 2:1 to 4:1 ratio.
* Erecting bird perches at a rate of 10 per acre to encourage natural FAW predation by birds.
Expected: Recommended practices include timely and uniform sowing, ridge and furrow planting, basal fertilizer application, trap crops, intercropping, bird perches and pheromone traps.
--------------------------------------------------------------------------------

Q02
Generated: I don't have enough information in the provided documents.
Expected: Napier grass or hybrid Napier can be planted as trap crops around mai

## Hybrid Answer Quality Evaluation

Evaluation method: Manual comparison against 15 reference answers.

Scoring:
- 1.0: Fully correct
- 0.5: Partially correct
- 0.0: Incorrect or failed to answer an answerable question

| Metric | Vector-only | Hybrid |
|---|---:|---:|
| Answer quality | 63.33% | 73.33% |
| Retrieval recall | 88.46% | 92.31% |

### Findings
- Hybrid retrieval improved answer quality by 10 percentage points.
- Q03 improved because the hybrid context contained all four
  recommended leguminous crops.
- Q11 remained incomplete despite retrieving both expected sources.
- Q02, Q12 and Q13 demonstrate remaining context and retrieval limitations.
- Both unsupported questions were correctly refused.

### Conclusion
Hybrid retrieval improved both source-level recall and answer
quality compared with the vector-only baseline. However, further
work is required to improve multi-document question answering.

## Consolidated Results Table

In [46]:
import pandas as pd

final_results = pd.DataFrame([
    {
        "Configuration": "Vector-only (200 chars)",
        "Chunks": 4420,
        "Retrieval Recall": "88.46%",
        "Answer Quality": "Not evaluated",
        "Avg Retrieval Latency": "Not recorded"
    },
    {
        "Configuration": "Vector-only (500 chars)",
        "Chunks": 1478,
        "Retrieval Recall": "88.46%",
        "Answer Quality": "70.00%",
        "Avg Retrieval Latency": "Not recorded"
    },
    {
        "Configuration": "Vector-only (1000 chars)",
        "Chunks": 702,
        "Retrieval Recall": "88.46%",
        "Answer Quality": "Not evaluated",
        "Avg Retrieval Latency": "Not recorded"
    },
    {
        "Configuration": "Hybrid BM25 + Vector",
        "Chunks": 1478,
        "Retrieval Recall": "92.31%",
        "Answer Quality": "73.33%",
        "Avg Retrieval Latency": "0.055 sec"
    },
    {
        "Configuration": "Vector + Cross-encoder",
        "Chunks": 1478,
        "Retrieval Recall": "92.31%",
        "Answer Quality": "Not evaluated",
        "Avg Retrieval Latency": "2.321 sec"
    }
])

display(final_results)

,Configuration,Chunks,Retrieval Recall,Answer Quality,Avg Retrieval Latency
0,Vector-only (200 chars),4420,88.46%,Not evaluated,Not recorded
1,Vector-only (500 chars),1478,88.46%,70.00%,Not recorded
2,Vector-only (1000 chars),702,88.46%,Not evaluated,Not recorded
3,Hybrid BM25 + Vector,1478,92.31%,73.33%,0.055 sec
4,Vector + Cross-encoder,1478,92.31%,Not evaluated,2.321 sec


## Final Conclusion

The experiments compared different chunk sizes, dense vector
retrieval, hybrid retrieval and cross-encoder reranking.

The 500-character chunk configuration was retained as the baseline.

Hybrid retrieval using BM25, vector search and Reciprocal Rank
Fusion achieved the best practical balance:

- Retrieval recall: 92.31%
- Answer quality: 73.33%
- Average retrieval latency: 0.055 seconds

Cross-encoder reranking also achieved 92.31% retrieval recall,
but its average latency was 2.321 seconds.

Therefore, hybrid retrieval was selected as the preferred
configuration for this agriculture-focused RAG system.

### Limitations
1. The evaluation used only 15 questions and 10 source documents.
2. Multi-document retrieval remains challenging.
3. Answer quality was evaluated manually.
4. Cross-encoder answer quality was not evaluated.
5. Latency measurements were collected in Google Colab.
6. The system has not been evaluated in a production environment.